In [197]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [198]:
def show (text):
    try:
        Console().print(text)
    except:
        print (text)

In [199]:
openai = OpenAI()

In [200]:
todos = []
completed = []

In [201]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [202]:
get_todo_report()

''

In [203]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False]* len(descriptions))
    return get_todo_report()

In [204]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index"
    Console().print(completion_notes)
    return get_todo_report()

In [205]:
todo, completed = [], []
create_todos(["Buy groceries", "FInish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: FInish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: FInish extra lab\nTodo #3: Eat banana\n'

In [206]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: FInish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: FInish extra lab\nTodo #3: Eat banana\n'

In [207]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Description"

            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [208]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from index 1) and return the full list",
    "parameters": {
        "properties": {
            "index": {
                "description": 'The 1-based index of the todo to mark as complete',
                "title": 'Index',
                "type": 'integer'
            },
            "completion_notes": {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [209]:
tools = [
    {'type': 'function', 'function': create_todos_json},
    {'type': 'function', 'function': mark_complete_json}
]

In [210]:
def handle_tools_calls(tool_calls):
    results =  []
    for  tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        # print (tool)
        # print (arguments)
        result = tool(**arguments) if tool else {}
        # print (result)
        results.append({'role': 'tool', "content": json.dumps(result), 'tool_call_id': tool_call.id})
    return results

In [211]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages = messages, tools = tools,reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason == 'tool_calls':
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tools_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [212]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A person is studying  in computer science in his bachelor degree.
How can he be millionaire in 10 years?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [213]:
todos, completed = [], []
loop(messages)

Todo #1: Define assumptions and a reasonable target (what ‘millionaire’ means and by when)
Todo #2: Lay out 2–3 viable pathways for a CS bachelor student to reach $1M net worth in 10 years
Todo #3: Quantify a baseline plan (salary growth, savings rate, investing) to see if $1M is feasible
Todo #4: Add high-upside options (startups, equity, side projects) and how to pursue them during/after college
Todo #5: List concrete year-by-year actions (skills, internships, jobs, networking, finances)
Todo #6: Add risk management and common pitfalls to avoid
Todo #7: Format the final response in Rich console markup, concise and actionable

Assume *millionaire* = **$1,000,000 net worth** (assets minus debts) by year 10 after starting now. Assume typical 
path: graduate in ~2 years, then 8 years work. Use conservative long-run index-fund return **7%/yr** (realistic 
average), and an alternative higher-return scenario via equity/entrepreneurship.

Todo #1: Define assumptions and a reasonable target (what ‘millionaire’ means and by when)
Todo #2: Lay out 2–3 viable pathways for a CS bachelor student to reach $1M net worth in 10 years
Todo #3: Quantify a baseline plan (salary growth, savings rate, investing) to see if $1M is feasible
Todo #4: Add high-upside options (startups, equity, side projects) and how to pursue them during/after college
Todo #5: List concrete year-by-year actions (skills, internships, jobs, networking, finances)
Todo #6: Add risk management and common pitfalls to avoid
Todo #7: Format the final response in Rich console markup, concise and actionable

Identified 3 paths:
1) **High-income employee + aggressive saving + index investing**.
2) **Employee + equity upside** (early-stage startup/RSUs) + investing.
3) **Founder/indie hacker** (SaaS/app/agency) + reinvest profits + investing.

Todo #1: Define assumptions and a reasonable target (what ‘millionaire’ means and by when)
Todo #2: Lay out 2–3 viable pathways for a CS bachelor student to reach $1M net worth in 10 years
Todo #3: Quantify a baseline plan (salary growth, savings rate, investing) to see if $1M is feasible
Todo #4: Add high-upside options (startups, equity, side projects) and how to pursue them during/after college
Todo #5: List concrete year-by-year actions (skills, internships, jobs, networking, finances)
Todo #6: Add risk management and common pitfalls to avoid
Todo #7: Format the final response in Rich console markup, concise and actionable

...

Risk controls: avoid high-interest debt, keep emergency fund, insure basics, diversify (index funds), don’t 
overconcentrate in employer stock, manage burnout, keep skills current; pitfalls: lifestyle inflation, speculative 
trading, underinvesting in career leverage.

Todo #1: Define assumptions and a reasonable target (what ‘millionaire’ means and by when)
Todo #2: Lay out 2–3 viable pathways for a CS bachelor student to reach $1M net worth in 10 years
Todo #3: Quantify a baseline plan (salary growth, savings rate, investing) to see if $1M is feasible
Todo #4: Add high-upside options (startups, equity, side projects) and how to pursue them during/after college
Todo #5: List concrete year-by-year actions (skills, internships, jobs, networking, finances)
Todo #6: Add risk management and common pitfalls to avoid
Todo #7: Format the final response in Rich console markup, concise and actionable

Prepared final answer using Rich markup with sections, numbers, and actionable bullets; concise but specific.

Todo #1: Define assumptions and a reasonable target (what ‘millionaire’ means and by when)
Todo #2: Lay out 2–3 viable pathways for a CS bachelor student to reach $1M net worth in 10 years
Todo #3: Quantify a baseline plan (salary growth, savings rate, investing) to see if $1M is feasible
Todo #4: Add high-upside options (startups, equity, side projects) and how to pursue them during/after college
Todo #5: List concrete year-by-year actions (skills, internships, jobs, networking, finances)
Todo #6: Add risk management and common pitfalls to avoid
Todo #7: Format the final response in Rich console markup, concise and actionable

Goal
Become a $1,000,000 net-worth person in 10 years (assets − debts). For most CS grads, this comes from a mix of:

[*]high income,
[*]high savings rate,
[*]consistent investing,
[*]and (optional) equity/side-business upside.


The “numbers” that make it work
If you invest in diversified index funds and average ~7%/year growth:

[*]Investing about $6,000/month for 10 years ≈ $1.0M.
[*]Investing $4,000/month for 10 years ≈ $690k (you’d need raises, a lump sum, or equity upside to reach $1M).

So the plan is to eventually reach (and sustain) roughly $5k–$6k/month invested by combining income growth + 
controlled lifestyle.

3 realistic paths for a CS student

Path A — High-income employee + aggressive saving + investing (most predictable)

[*]Target top-paying markets/companies (Big Tech, strong fintech, top startups, remote US/EU pay if possible).
[*]Live on ~40–60% of take-home, invest the rest automatically.
[*]Job-hop strategically early (every 1.5–3 years) to grow compensation faster.


Path B — Employee + equity upside (higher risk, higher reward)

[*]Join a startup with meaningful equity OR a public company with RSUs.
[*]Treat equity as a bonus: sell/diversify as it vests (don’t let one stock be your whole net worth).
[*]Still do consistent index investing so you’re not depending on “one big win.”


Path C — Founder/side-business (highest upside)

[*]Build SaaS/app/plugins/templates/APIs, or a niche agency/productized service.
[*]Aim for progression: $500/month → $2k → $10k+ MRR over years.
[*]Reinvest profits into growth, then diversify profits into index funds.


10-year action plan (what to do each phase)

Years 0–2 (while in bachelor’s): build “career leverage”

[*]Pick a high-demand lane: Backend (Java/Go), Cloud/DevOps, Data/ML engineering, Mobile, Security, or strong 
Full-stack.
[*]Portfolio that gets interviews: 2–3 real projects (deployed), clean GitHub, readable README, tests/basic CI.
[*]Internships: apply broadly; referrals matter—network with seniors/alumni weekly.
[*]Interview skills: DSA + system design basics (this directly increases income).
[*]Money rule: avoid consumer debt; build a small emergency fund.


Years 3–5 (first job years): maximize income growth + savings rate

[*]Choose roles with strong learning + brand + compensation.
[*]Negotiate every offer; keep a “brag document” to justify promotions/raises.
[*]Automate investing each paycheck (broad index funds/retirement accounts if available).
[*]Keep lifestyle modest even as salary rises (this is where most people fail).


Years 6–10 (senior years): scale compensation or add ownership

[*]Push to senior/lead or specialized high-paying niche.
[*]Consider switching to companies with large RSU packages or joining a strong startup with equity.
[*]If doing a side business: focus on distribution (SEO, content, partnerships, outbound) more than extra features.
[*]By now, target investing around $5k–$6k/month average (or equivalent via bonuses/equity/lump sums).


Where the millionaire outcome usually comes from

[*]Compounding + consistency (automatic investing for a decade).
[*]Income growth (skills + interview ability + job moves).
[*]Avoiding lifestyle inflation (keeping fixed costs low).
[*]Optional upside (equity or a side business can “fill the gap” if monthly investing is lower early on).


Risk management (so you don’t lose progress)

[*]Keep 3–6 months expenses as an emergency fund.
[*]Avoid day trading/crypto gambling as your main plan.
[*]Don’t let employer stock become most of your net worth; diversify.
[*]Protect health and avoid burnout (career income is the engine).


A simple rule to follow
Spend like a student for a few years after graduating, earn like an engineer, invest like a boring long-term 
investor.